# Time-Series Classification Using Topological Features  

In this notebook we will investigate the use of persistent homology features for time-series classification, and compare them to conventional statistical features.

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root  / "src"))

from tda_features import delay_embed, extract_H1_features


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Construction of a Classification Dataset  

We have seen in the previous notebook that topological features can distinguish a periodic signal from a shuffled collection of the corresponding observations.  

Next, we want to investigate whether the five normalized H1 features indicated at the end of the previosu notebook can be used to systematically distinguish these two using supervised machine learning classification methods.  
We construct a dataset of paired signals. At each repetition a noisy sine signal is generated, and then a corresponding shuffled signal is constructed with the same observations.   
As seen before, the two signals inside of each pair have the same empirical distribution (mean and variance in particular), but different temporal structure.  

The following parameters are used:  
- Number of paired realizations: $100$;
- Numer of observations per signal: $500$;
- Gaussian noise standard deviation: $\sigma = 0.2$;
- Delay parameter: $\tau = 25$;
- Embedding dimension: $d = 3$;
- Prominence treshold: $0.1$

For each generated signal, the corresponding delay embedding is generated, and from that the persistent homology features are extracted.  
Each observation is assigned a class label (0 for shuffled, 1 for periodic), and a pair id is assigned to each pair of signals.

In [3]:
t = np.linspace(0, 10 * np.pi, 500, endpoint = False)
clean_signal = np.sin(t)

In [7]:
n_pairs = 100
rows = []
sigma = 0.2

rng = np.random.default_rng(2408)

for pair_id in range(n_pairs):

    noise = rng.normal(0, sigma, size = len(clean_signal))

    periodic_signal = clean_signal + noise
    shuffled_signal = rng.permutation(periodic_signal)

    periodic_embedding = delay_embed(periodic_signal, tau=25, d=3)
    shuffled_embedding = delay_embed(shuffled_signal, tau=25, d=3)

    periodic_features = extract_H1_features(periodic_embedding)
    shuffled_features = extract_H1_features(shuffled_embedding)
    
    periodic_features["label"] = 1
    periodic_features["pair_id"] = pair_id

    shuffled_features["label"] = 0
    shuffled_features["pair_id"] = pair_id

    rows.append(periodic_features)
    rows.append(shuffled_features)
    

In [8]:
dataframe = pd.DataFrame(rows)

print(dataframe.shape)
print(dataframe.head())

(200, 7)
     max_H1  second_H1  total_H1_persistence  prominent_H1_features  \
0  0.232153   0.040409              1.662104                      1   
1  0.067987   0.056519              2.882819                      0   
2  0.263603   0.039391              1.596972                      1   
3  0.062092   0.058250              2.990475                      0   
4  0.241457   0.031153              1.707876                      1   

   persistence_entropy  label  pair_id  
0             4.479697      1        0  
1             4.889494      0        0  
2             4.371877      1        1  
3             4.942093      0        1  
4             4.539097      1        2  


In [10]:
# Dataset structure
print(dataframe.shape)
print(dataframe["label"].value_counts())

# Pair integrity
pair_labels = dataframe.groupby("pair_id")["label"].agg(list)
print("Valid pairs:", pair_labels.apply(
    lambda labels: sorted(labels) == [0, 1]
).all())
print("Number of pairs:", len(pair_labels))

# Numerical validity
feature_cols = dataframe.columns.drop(["label", "pair_id"])
print("All features finite:",
      np.isfinite(dataframe[feature_cols].to_numpy()).all())

(200, 7)
label
1    100
0    100
Name: count, dtype: int64
Valid pairs: True
Number of pairs: 100
All features finite: True
